# Detectron2 数据增强 Data Augmentation 完整Demo

文档来源：Detectron2 0.6 Documentation — Data Augmentation

本 Notebook 覆盖教程全部 4 大块内容：
1. 基础用法 Basic Usage
2. 编写新的增强 Write New Augmentations
3. 高级用法‑自定义变换策略 Custom transform strategy
4. 高级用法‑几何逆变换 Geometrically invert the transform
5. 高级用法‑新增数据类型 Add new data types
6. 高级用法‑扩展 T.AugInput

## 环境检测

运行前先检查本机的 Python / PyTorch / CUDA / detectron2 版本，确认环境匹配（CUDA 与 PyTorch、detectron2 wheel 需对应）。

In [ ]:
import sys
print("Python:       ", sys.version.split()[0])

try:
    import torch
    print("PyTorch:      ", torch.__version__)
    print("CUDA (torch): ", torch.version.cuda)
    print("GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch:       未安装")

try:
    import detectron2
    print("detectron2:   ", detectron2.__version__)
except ImportError:
    print("detectron2:    未安装")

## 0. 环境准备

【Colab 取消注释执行】安装 detectron2，注意匹配 cuda、torch 版本。以下命令先注释保留，确认上一节环境检测无误后再按需执行。

In [ ]:
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html

import numpy as np
from detectron2.data import transforms as T
import matplotlib.pyplot as plt

## 构造模拟测试数据

无需真实数据集，构造模拟图像、bbox、语义分割、关键点、多边形

In [ ]:
# H,W 图像高宽
H, W = 600, 800
# 模拟RGB图像 (H,W,3) uint8
image = np.random.randint(low=0, high=255, size=(H, W, 3), dtype=np.uint8)

# bboxes: [x1,y1,x2,y2]
boxes = np.array([[50, 50, 200, 180], [300, 200, 450, 350]], dtype=np.float32)

# 模拟语义分割单通道图
sem_seg = np.random.randint(low=0, high=20, size=(H, W), dtype=np.uint8)

# 模拟关键点坐标 (N,2) x,y
keypoints_xy = np.array([[120, 100], [140, 110], [330, 220]], dtype=np.float32)

# 模拟多边形标注
polygons = np.array([[60, 60, 190, 60, 190, 170, 60, 170]], dtype=np.float32)

# 旋转框示例
rotated_boxes = np.array([[100, 100, 80, 60, 15]], dtype=np.float32)

print(f"image shape: {image.shape}")
print(f"boxes:\n{boxes}")

## 1. Basic Usage 基础用法

核心对象：T.AugmentationList、T.AugInput、T.Transform

In [ ]:
from detectron2.data import transforms as T

# 1.1 定义一组增强流水线
augs = T.AugmentationList([
    T.RandomBrightness(0.9, 1.1),    # 随机亮度扰动
    T.RandomFlip(prob=0.5),          # 0.5概率水平翻转
    T.RandomCrop("absolute", (640, 640))  # 绝对尺寸裁剪到640x640
])  # type: T.AugmentationList

# 1.2 构造AugInput，传入需要同步增强的数据
# image必填，boxes/sem_seg为可选
input = T.AugInput(image, boxes=boxes, sem_seg=sem_seg)

# 1.3 执行增强；原地修改input，返回Transform记录执行的变换操作
transform = augs(input)   # type: T.Transform

# 获取增强之后的数据
image_transformed = input.image
sem_seg_transformed = input.sem_seg

# 对于不在AugInput内部的额外数据，手动使用transform对象做变换
image2_transformed = transform.apply_image(image)
polygons_transformed = transform.apply_polygons(polygons)

print("✅基础用法执行完成")
print(f"原始图像shape {image.shape} ->增强后 {image_transformed.shape}")

## 2. Write New Augmentations 编写自定义增强算子

继承 T.Augmentation，重写 get_transform() 返回 T.Transform 对象

In [ ]:
# 自定义颜色增强
class MyColorAugmentation(T.Augmentation):
    def get_transform(self, image):
        # 随机系数
        r = np.random.rand(2)
        # 返回颜色变换算子
        return T.ColorTransform(lambda x: x * r[0] + r[1] * 10)

# 自定义Resize增强
class MyCustomResize(T.Augmentation):
    def get_transform(self, image):
        old_h, old_w = image.shape[:2]
        new_h = int(old_h * np.random.rand())
        new_w = int(old_w * 1.5)
        return T.ResizeTransform(old_h, old_w, new_h, new_w)

# 自定义裁剪，可以同时使用image与sem_seg信息做裁剪决策
class MyCustomCrop(T.Augmentation):
    def get_transform(self, image, sem_seg):
        h, w = image.shape[:2]
        return T.CropTransform(x0=10, y0=10, x1=w-50, y1=h-50)


# 测试自定义Resize
aug_custom_resize = MyCustomResize()
inp_test = T.AugInput(image.copy())
trans_resize = aug_custom_resize(inp_test)
img_resized = inp_test.image
print(f"✅自定义Resize，原图 {image.shape} → {img_resized.shape}")

## 3. Advanced Usage 高级用法

### 3.1 Custom transform strategy 自定义变换策略，关键点处理示例

In [ ]:
# 模拟增强
augs_kp = T.AugmentationList([
    T.RandomFlip(prob=1.0),  # 强制翻转，方便观察关键点逻辑
])
input_kp = T.AugInput(image.copy())
transform = augs_kp(input_kp)  # type:T.TransformList

# 对关键点坐标执行变换
keypoints_xy_t = transform.apply_coords(keypoints_xy.copy())

# 获取全部变换列表，判断是否发生奇数次水平翻转
transforms = T.TransformList([transform]).transforms
do_hflip = sum(isinstance(t, T.HFlipTransform) for t in transforms) % 2 == 1

# 模拟人体关键点左右眼索引映射，翻转时交换左眼右眼
flip_indices_mapping = [1, 0, 2]
if do_hflip:
    keypoints_xy_t = keypoints_xy_t[flip_indices_mapping]

print(f"✅关键点增强完成，是否水平翻转：{do_hflip}")
print(f"原始关键点：{keypoints_xy}")
print(f"变换后关键点：{keypoints_xy_t}")

# ---- 逐变换步骤检查关键点可见性示例 ----
keypoints_copy = keypoints_xy.copy()
visibility = np.ones(keypoints_copy.shape[0], dtype=bool)
for t in transform.transforms:
    keypoints_copy = t.apply_coords(keypoints_copy)
    # 判断坐标是否落在图像边界内
    visibility &= ((keypoints_copy >= [0, 0]) & (keypoints_copy <= [W, H])).all(axis=1)
print(f"关键点可见性：{visibility}")

### 3.2 Geometrically invert the transform 逆变换

推理时图像经过增强，预测结果需要逆变换映射回原图

In [ ]:
augs_inv = T.AugmentationList([
    T.RandomFlip(prob=1.0),
    T.Resize((400,400))
])
input_inv = T.AugInput(image.copy())
trans = augs_inv(input_inv)

# 模拟模型预测得到mask
pred_mask = np.random.randint(0, 2, size=input_inv.image.shape[:2], dtype=np.uint8)

# 获取逆变换对象
inv_transform = trans.inverse()
# 将预测mask映射回原始图像坐标系
pred_mask_orig = inv_transform.apply_segmentation(pred_mask)

print(f"✅逆变换完成，预测mask增强后 {pred_mask.shape} →原图尺寸 {pred_mask_orig.shape}")

### 3.3 Add new data types：注册新的数据类型（rotated_box旋转框）

In [ ]:
# 为HFlipTransform注册 rotated_boxes 的处理函数
@T.HFlipTransform.register_type("rotated_boxes")
def hflip_rotated_box(flip_transform: T.HFlipTransform, rotated_boxes: np.ndarray):
    """
    rotated_boxes shape [N,5] xc,yc,w,h,angle
    水平翻转：xc = width - xc
    """
    w_img = flip_transform.width
    out = rotated_boxes.copy()
    out[:,0] = w_img - out[:,0]
    out[:,4] = -out[:,4]  # 角度取反
    return out

# 实例化水平翻转变换
t = T.HFlipTransform(width=W)
rotated_boxes_after = t.apply_rotated_boxes(rotated_boxes)

print("✅旋转框注册测试")
print(f"原始rotated_boxes:\n{rotated_boxes}")
print(f"翻转后rotated_boxes:\n{rotated_boxes_after}")

### 3.4 Extend T.AugInput 扩展AugInput

重写transform方法，实现字段之间相互依赖的增强逻辑

In [ ]:
class MyAugInput(T.AugInput):
    def transform(self, transform: T.Transform):
        super().transform(transform)
        # 示例：根据变换后的mask，后处理bbox（自定义依赖逻辑）
        if self.sem_seg is not None and self.boxes is not None:
            # 这里仅演示占位，业务上写自己的联动逻辑
            pass

# 使用自定义AugInput
my_input = MyAugInput(image.copy(), boxes=boxes, sem_seg=sem_seg)
aug_test = T.RandomFlip(prob=0.5)
trans_my = aug_test(my_input)
print("✅自定义AugInput执行完成")

# 总结

1. T.Augmentation：定义增强策略（policy），输出T.Transform
2. T.Transform：保存实际变换操作，提供apply_xxx系列接口
3. T.AugInput：存放待增强的图像、box、seg等输入
4. 高级特性：逆变换inverse()、注册新数据类型、自定义AugInput实现跨字段联动